In [1]:
#Observation 1

In [2]:
from pathlib import Path
from collections import Counter
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv"

# Canonical style names used in the paper
STYLE_CANON = ["Community", "Custom", "GMD", "Third Party", "Real Device", "Unclassified"]

# Tokens expected inside Exec_Env_Style (robust mapping)
TOKEN_MAP = {
    "emu_community": "Community",
    "community": "Community",
    "emu_custom": "Custom",
    "custom": "Custom",
    "emu_gmd": "GMD",
    "gmd": "GMD",
    "third-party": "Third Party",
    "third_party": "Third Party",
    "third party": "Third Party",
    "real_device": "Real Device",
    "real device": "Real Device",
    "unclassified": "Unclassified",
}

def norm_bool(x) -> bool:
    s = str(x).strip().lower()
    return s in {"1", "true", "t", "yes", "y"}

def pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns exist: {candidates}")

def parse_style_set(s: str) -> set[str]:
    """Parse comma-separated Exec_Env_Style into canonical style set."""
    if s is None:
        return set()
    raw = [p.strip() for p in str(s).split(",") if p.strip()]
    out = set()
    for tok in raw:
        k = tok.strip().lower()
        out.add(TOKEN_MAP.get(k, None) or TOKEN_MAP.get(k.replace("-", "_"), None) or tok)
    # Keep only canonical names if possible
    out2 = set()
    for v in out:
        vv = str(v).strip()
        if vv in STYLE_CANON:
            out2.add(vv)
        else:
            # last attempt normalize unknown
            kk = vv.lower()
            if kk in TOKEN_MAP:
                out2.add(TOKEN_MAP[kk])
    return out2

# ============================================================
# Load and filter adopters
# ============================================================
p = Path(CSV_PATH)
if not p.exists():
    raise FileNotFoundError(p)

df = pd.read_csv(p, dtype=str).fillna("")
col_adopt = pick_col(df, ["instru_t_ci_signal"])
col_style = pick_col(df, ["Exec_Env_Style", "exec_env_style", "execution_environment_style"])

df["adopt"] = df[col_adopt].apply(norm_bool)
ad = df[df["adopt"]].copy()

# style sets
ad["style_set"] = ad[col_style].apply(parse_style_set)

# ============================================================
# Any-use prevalence
# ============================================================
any_use = {}
for style in STYLE_CANON:
    any_use[style] = int(ad["style_set"].apply(lambda ss: style in ss).sum())

any_tbl = pd.DataFrame(
    [{"Style": s, "Repos": any_use[s], "Share": any_use[s] / len(ad) if len(ad) else 0.0}
     for s in STYLE_CANON]
)
any_tbl["Share"] = (any_tbl["Share"] * 100).round(1).astype(str) + "%"

# ============================================================
# Single vs mixed
# ============================================================
ad["n_styles"] = ad["style_set"].apply(len)
single = ad[ad["n_styles"] == 1].copy()
mixed  = ad[ad["n_styles"] >= 2].copy()

single_tbl = pd.DataFrame([
    {"Group": "Single-style adopters", "Repos": len(single), "Share": f"{(len(single)/len(ad)*100):.1f}%"},
    {"Group": "Mixed-style adopters",  "Repos": len(mixed),  "Share": f"{(len(mixed)/len(ad)*100):.1f}%"},
])

# ============================================================
# Single-style per style
# ============================================================
single_counts = Counter()
for ss in single["style_set"]:
    only = next(iter(ss))
    single_counts[only] += 1

single_style_tbl = pd.DataFrame(
    [{"Style": s, "Repos": single_counts.get(s, 0), "Share": single_counts.get(s, 0)/len(ad) if len(ad) else 0.0}
     for s in STYLE_CANON]
)
single_style_tbl["Share"] = (single_style_tbl["Share"] * 100).round(1).astype(str) + "%"

# ============================================================
# Mixed combinations (top)
# ============================================================
combo_counts = Counter()
for ss in mixed["style_set"]:
    combo_counts[tuple(sorted(ss))] += 1

combo_tbl = pd.DataFrame(
    [{"Combination": " + ".join(k), "Repos": v, "Share": f"{(v/len(ad)*100):.1f}%"} 
     for k, v in combo_counts.most_common(10)]
)

# ============================================================
# Print
# ============================================================
print("\n=== Observation 2.1 (Any-use prevalence among adopters) ===")
print(any_tbl.to_string(index=False))

print("\n=== Observation 2.1 (Single vs Mixed adopters) ===")
print(single_tbl.to_string(index=False))

print("\n=== Observation 2.1 (Single-style counts by style; share over ALL adopters) ===")
print(single_style_tbl.to_string(index=False))

print("\n=== Observation 2.1 (Top mixed-style combinations) ===")
print(combo_tbl.to_string(index=False))

print(f"\nAdopters N = {len(ad)}")



=== Observation 2.1 (Any-use prevalence among adopters) ===
       Style  Repos Share
   Community    241 50.2%
      Custom    160 33.3%
         GMD     16  3.3%
 Third Party     32  6.7%
 Real Device      2  0.4%
Unclassified      0  0.0%

=== Observation 2.1 (Single vs Mixed adopters) ===
                Group  Repos Share
Single-style adopters    409 85.2%
 Mixed-style adopters     21  4.4%

=== Observation 2.1 (Single-style counts by style; share over ALL adopters) ===
       Style  Repos Share
   Community    224 46.7%
      Custom    149 31.0%
         GMD      7  1.5%
 Third Party     27  5.6%
 Real Device      2  0.4%
Unclassified      0  0.0%

=== Observation 2.1 (Top mixed-style combinations) ===
            Combination  Repos Share
        Community + GMD      8  1.7%
     Community + Custom      7  1.5%
   Custom + Third Party      3  0.6%
Community + Third Party      2  0.4%
           Custom + GMD      1  0.2%

Adopters N = 480


In [ ]:
#Observation 2.

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np
import math
from scipy.stats import chi2_contingency

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv"

# Keep your paper style names (no dataset-label talk)
STYLE_MAP = {
    "Emu_Community": "Community",
    "Emu_Custom": "Custom",
    "Emu_GMD": "GMD",
    "Third-Party": "Third Party",
    "Real Device": "Real Device",
    "Unclassified": "Unclassified",
    "": "Unclassified",
}

# CI service columns in YOUR dataset
SERVICE_COLS = {
    "GitHub Actions": "github_actions",
    "Travis CI": "travis_ci",
    "CircleCI": "circle_ci",
    "GitLab CI": "gitlab",
}

# ============================================================
# Helpers
# ============================================================
def is_present(x) -> bool:
    """
    Treat CI service as present if value is:
      - boolean True
      - a truthy string ("true", "yes", ...)
      - a numeric/count string > 0 (e.g., "2")
      - a numeric value > 0
    """
    if pd.isna(x):
        return False
    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    s = str(x).strip().lower()
    if s in {"true", "t", "yes", "y"}:
        return True
    if s in {"false", "f", "no", "n", ""}:
        return False

    # numeric/count
    try:
        return float(s) > 0
    except ValueError:
        return False

def parse_style_set(s: str):
    parts = [p.strip() for p in str(s).split(",") if p.strip()]
    if not parts:
        return {"Unclassified"}
    mapped = {STYLE_MAP.get(p, p) for p in parts}
    return mapped

def cramers_v_from_table(tab: pd.DataFrame) -> float:
    chi2, p, dof, exp = chi2_contingency(tab)
    n = tab.to_numpy().sum()
    r, c = tab.shape
    return math.sqrt((chi2 / n) / (min(r - 1, c - 1)))

# ============================================================
# Load
# ============================================================
p = Path(CSV_PATH)
if not p.exists():
    raise FileNotFoundError(f"CSV not found: {p}")

df = pd.read_csv(p).fillna("")

# adopt flag (in your dataset this is already repo-level)
df["adopt"] = df["instru_t_ci_signal"].apply(lambda x: str(x).strip().lower() in {"true", "1", "t", "yes", "y"})
ad = df[df["adopt"]].copy()

# styles (multi-label)
ad["style_set"] = ad["Exec_Env_Style"].apply(parse_style_set)
ad["n_styles"] = ad["style_set"].apply(len)

# Option A comparisons: single-style repos only
single = ad[ad["n_styles"] == 1].copy()
single["style"] = single["style_set"].apply(lambda ss: next(iter(ss)))

# Skip Real Device for tests (tiny N), but keep it if you want in prevalence elsewhere
single = single[single["style"] != "Real Device"].copy()

# normalize CI service presence from count-columns
for pretty, col in SERVICE_COLS.items():
    if col not in single.columns:
        raise KeyError(f"Missing expected column '{col}' for {pretty}. موجود columns: {list(single.columns)}")
    single[col] = single[col].apply(is_present)

# ============================================================
# Build per-style table + chi-square per service
# ============================================================
style_sizes = single.groupby("style").size().sort_values(ascending=False)

rows = []
test_rows = []

for pretty, col in SERVICE_COLS.items():
    # counts per style
    counts = single.groupby("style")[col].sum().reindex(style_sizes.index).fillna(0).astype(int)
    rates = (counts / style_sizes * 100).round(1)

    # contingency: style x {False, True}
    tab = pd.crosstab(single["style"], single[col]).reindex(index=style_sizes.index, fill_value=0)
    tab = tab.reindex(columns=[False, True], fill_value=0)

    # guard: if all rows are only True or only False, chi2 is meaningless
    if (tab[True].sum() == 0) or (tab[False].sum() == 0):
        chi2, pval, v = 0.0, 1.0, 0.0
    else:
        chi2, pval, dof, exp = chi2_contingency(tab)
        v = cramers_v_from_table(tab)

    test_rows.append({
        "CI service": pretty,
        "chi2": chi2,
        "p": pval,
        "Cramer's V": v,
    })

# Pretty output table (counts + %)
out = pd.DataFrame({"N": style_sizes})
for pretty, col in SERVICE_COLS.items():
    cnt = single.groupby("style")[col].sum().reindex(style_sizes.index).fillna(0).astype(int)
    out[pretty] = [f"{c} ({c/n*100:.1f}%)" for c, n in zip(cnt, style_sizes)]

print("\n=== Observation 2.4: CI service footprint by execution style (single-style only) ===")
print(out)

tests = pd.DataFrame(test_rows).sort_values("p")
print("\n=== Chi-square tests (service presence vs style) ===")
print(tests)







=== Observation 2.4: CI service footprint by execution style (single-style only) ===
                N GitHub Actions    Travis CI    CircleCI  GitLab CI
style                                                               
Community     224   224 (100.0%)     6 (2.7%)    3 (1.3%)   1 (0.4%)
Custom        149     23 (15.4%)  116 (77.9%)  23 (15.4%)   3 (2.0%)
Unclassified   50     26 (52.0%)   20 (40.0%)    1 (2.0%)   4 (8.0%)
Third Party    27     17 (63.0%)     2 (7.4%)   9 (33.3%)   0 (0.0%)
GMD             7      6 (85.7%)     0 (0.0%)    0 (0.0%)  1 (14.3%)

=== Chi-square tests (service presence vs style) ===
       CI service        chi2             p  Cramer's V
0  GitHub Actions  285.721103  1.301345e-60    0.790702
1       Travis CI  246.712648  3.324275e-52    0.734747
2        CircleCI   52.014203  1.370052e-10    0.337367
3       GitLab CI   18.154180  1.151344e-03    0.199311


In [ ]:
#observation 3

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from bisect import bisect_left, bisect_right

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv"

STYLE_CANON = ["Community", "Custom", "GMD", "Third Party", "Real Device", "Unclassified"]
TOKEN_MAP = {
    "emu_community": "Community", "community": "Community",
    "emu_custom": "Custom", "custom": "Custom",
    "emu_gmd": "GMD", "gmd": "GMD",
    "third-party": "Third Party", "third_party": "Third Party", "third party": "Third Party",
    "real_device": "Real Device", "real device": "Real Device",
    "unclassified": "Unclassified",
}

def norm_bool(x) -> bool:
    s = str(x).strip().lower()
    return s in {"1", "true", "t", "yes", "y"}

def pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns exist: {candidates}")

def parse_style_set(s: str) -> set[str]:
    raw = [p.strip() for p in str(s).split(",") if p.strip()]
    out = set()
    for tok in raw:
        k = tok.strip().lower()
        out.add(TOKEN_MAP.get(k, None) or TOKEN_MAP.get(k.replace("-", "_"), None) or tok)
    return {v for v in out if v in STYLE_CANON}

def to_num(series: pd.Series):
    return pd.to_numeric(series, errors="coerce").dropna().to_numpy()

def cliffs_delta(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cliff's delta for a vs b:
      delta = (#{a>b} - #{a<b}) / (len(a)*len(b))
    Efficient O(n log n) using sorting + binary search.
    """
    a = np.asarray(a); b = np.asarray(b)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    b_sorted = np.sort(b)
    gt = lt = 0
    for x in a:
        lt += bisect_left(b_sorted, x)             # b < x
        gt += (len(b_sorted) - bisect_right(b_sorted, x))  # b > x
    return (lt - gt) / (len(a) * len(b))

# ============================================================
# Load dataset and build Community vs Custom (single-style only)
# ============================================================
p = Path(CSV_PATH)
df = pd.read_csv(p, dtype=str).fillna("")

col_adopt = pick_col(df, ["instru_t_ci_signal"])
col_style = pick_col(df, ["Exec_Env_Style", "exec_env_style"])

# metrics (robust candidate lists)
col_pr  = pick_col(df, ["pull_requests", "pull_request", "prs", "num_pull_requests"])
col_cm  = pick_col(df, ["commits_GitAPI", "commits", "num_commits"])
col_con = pick_col(df, ["contributors", "num_contributors"])
col_age = pick_col(df, ["repo_age", "age_years", "repository_age_years", "repo_age_years"])
col_iss = pick_col(df, ["open_issues_count", "open_issues", "issues_open"])

df["adopt"] = df[col_adopt].apply(norm_bool)
ad = df[df["adopt"]].copy()

ad["style_set"] = ad[col_style].apply(parse_style_set)
ad["n_styles"]  = ad["style_set"].apply(len)

single = ad[ad["n_styles"] == 1].copy()
single["style"] = single["style_set"].apply(lambda ss: next(iter(ss)))

comm = single[single["style"] == "Community"]
cust = single[single["style"] == "Custom"]

# ============================================================
# MWU + Cliff's delta
# ============================================================
tests = [
    (col_pr,  "Pull requests", "greater"),
    (col_cm,  "Commits",       "greater"),
    (col_con, "Contributors",  "greater"),
    (col_iss, "Open issues",   "greater"),
    (col_age, "Repo age (yrs)","less"),  # Community < Custom
]

rows = []
for col, name, alt in tests:
    a = to_num(comm[col])
    b = to_num(cust[col])

    med_a = float(np.median(a)) if len(a) else np.nan
    med_b = float(np.median(b)) if len(b) else np.nan

    # Mann-Whitney
    res = mannwhitneyu(a, b, alternative=alt)

    # Cliff's delta always computed as Community vs Custom (sign will reflect direction)
    d = cliffs_delta(a, b)

    rows.append({
        "Metric (direction)": f"{name} (Comm. {'>' if alt=='greater' else '<'} Cust.)",
        "Median (Comm.)": med_a,
        "Median (Cust.)": med_b,
        "p (1-sided)": res.pvalue,
        "Cliff's δ": d,
    })

out = pd.DataFrame(rows)

print("\n=== Observation 2.3 (Single-style only) ===")
print(f"N(Community) = {len(comm)}  |  N(Custom) = {len(cust)}")
print(out.to_string(index=False))



=== Observation 2.3 (Single-style only) ===
N(Community) = 224  |  N(Custom) = 149
            Metric (direction)  Median (Comm.)  Median (Cust.)  p (1-sided)  Cliff's δ
 Pull requests (Comm. > Cust.)          148.50           13.00 5.766800e-15   0.471806
       Commits (Comm. > Cust.)          593.50          163.00 8.890936e-12   0.410924
  Contributors (Comm. > Cust.)           10.00            4.00 1.603937e-06   0.283737
   Open issues (Comm. > Cust.)           17.50           11.00 4.965844e-03   0.157508
Repo age (yrs) (Comm. < Cust.)            5.13            9.49 1.224696e-19  -0.549527


In [ ]:
#Observation 4

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv"

STYLE_CANON = ["Community", "Custom", "GMD", "Third Party", "Real Device", "Unclassified"]
TOKEN_MAP = {
    "emu_community": "Community", "community": "Community",
    "emu_custom": "Custom", "custom": "Custom",
    "emu_gmd": "GMD", "gmd": "GMD",
    "third-party": "Third Party", "third_party": "Third Party", "third party": "Third Party",
    "real_device": "Real Device", "real device": "Real Device",
    "unclassified": "Unclassified",
}

def norm_bool(x) -> bool:
    s = str(x).strip().lower()
    return s in {"1", "true", "t", "yes", "y"}

def pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns exist: {candidates}")

def parse_style_set(s: str) -> set[str]:
    raw = [p.strip() for p in str(s).split(",") if p.strip()]
    out = set()
    for tok in raw:
        k = tok.strip().lower()
        out.add(TOKEN_MAP.get(k, None) or TOKEN_MAP.get(k.replace("-", "_"), None) or tok)
    return {v for v in out if v in STYLE_CANON}

def cramers_v(chi2, n, r, c):
    denom = n * (min(r - 1, c - 1))
    return np.sqrt(chi2 / denom) if denom > 0 else 0.0

# ============================================================
# Load & filter single-style adopters
# ============================================================
p = Path(CSV_PATH)
df = pd.read_csv(p, dtype=str).fillna("")

col_adopt = pick_col(df, ["instru_t_ci_signal"])
col_style = pick_col(df, ["Exec_Env_Style", "exec_env_style"])

# CI service presence columns (adjust candidates if your dataset differs)
col_gha    = pick_col(df, ["github_actions", "GitHub_Actions", "gha", "has_github_actions"])
col_travis = pick_col(df, ["travis_ci", "Travis_CI", "travis", "has_travis"])
col_circle = pick_col(df, ["circle_ci", "circleci", "CircleCI", "has_circleci"])
col_gitlab = pick_col(df, ["gitlab", "gitlab_ci", "GitLab_CI", "gitlabci", "has_gitlab_ci"])


df["adopt"] = df[col_adopt].apply(norm_bool)
ad = df[df["adopt"]].copy()

ad["style_set"] = ad[col_style].apply(parse_style_set)
ad["n_styles"]  = ad["style_set"].apply(len)

single = ad[ad["n_styles"] == 1].copy()
single["style"] = single["style_set"].apply(lambda ss: next(iter(ss)))

# Exclude Real Device from tests (too small)
single = single[single["style"] != "Real Device"].copy()

# normalize services to boolean
single["GHA"]    = single[col_gha].apply(norm_bool)
single["Travis"] = single[col_travis].apply(norm_bool)
single["Circle"] = single[col_circle].apply(norm_bool)
single["GitLab"] = single[col_gitlab].apply(norm_bool)

# ============================================================
# Build footprint table (counts + % within style)
# ============================================================
def pct(n, d): 
    return (100.0 * n / d) if d else 0.0

rows = []
for style, g in single.groupby("style"):
    n = len(g)
    rows.append({
        "Style": style, "N": n,
        "GitHub Actions": f"{g['GHA'].sum()} ({pct(g['GHA'].sum(), n):.1f}%)",
        "Travis CI":      f"{g['Travis'].sum()} ({pct(g['Travis'].sum(), n):.1f}%)",
        "CircleCI":       f"{g['Circle'].sum()} ({pct(g['Circle'].sum(), n):.1f}%)",
        "GitLab CI":      f"{g['GitLab'].sum()} ({pct(g['GitLab'].sum(), n):.1f}%)",
    })

footprint = pd.DataFrame(rows).sort_values("N", ascending=False)

print("\n=== Observation 2.4 (Single-style adopters; Real Device excluded) ===")
print(f"N(single-style, no Real Device) = {len(single)}")
print("\n--- CI service footprint by style ---")
print(footprint.to_string(index=False))

# ============================================================
# χ² tests per service: Style x {service present/absent}
# ============================================================
tests = []
for svc_col, svc_name in [("GHA", "GitHub Actions"), ("Travis", "Travis CI"),
                          ("Circle", "CircleCI"), ("GitLab", "GitLab CI")]:
    ct = pd.crosstab(single["style"], single[svc_col])
    chi2, pval, dof, _ = chi2_contingency(ct)
    V = cramers_v(chi2, n=ct.to_numpy().sum(), r=ct.shape[0], c=ct.shape[1])
    tests.append({"Service": svc_name, "Chi2": chi2, "dof": dof, "p": pval, "Cramér's V": V})

tests_df = pd.DataFrame(tests).sort_values("Cramér's V", ascending=False)

print("\n--- Style x Service χ² tests ---")
print(tests_df.to_string(index=False, float_format=lambda x: f"{x:.4g}"))



=== Observation 2.4 (Single-style adopters; Real Device excluded) ===
N(single-style, no Real Device) = 407

--- CI service footprint by style ---
      Style   N GitHub Actions   Travis CI   CircleCI GitLab CI
  Community 224     61 (27.2%)    6 (2.7%)   3 (1.3%)  1 (0.4%)
     Custom 149      10 (6.7%) 114 (76.5%) 23 (15.4%)  3 (2.0%)
Third Party  27       1 (3.7%)    1 (3.7%)  8 (29.6%)  0 (0.0%)
        GMD   7       0 (0.0%)    0 (0.0%)   0 (0.0%) 1 (14.3%)

--- Style x Service χ² tests ---
       Service  Chi2  dof         p  Cramér's V
     Travis CI 246.3    3 4.217e-53      0.7779
      CircleCI 40.76    3 7.351e-09      0.3165
GitHub Actions 31.47    3  6.76e-07      0.2781
     GitLab CI 12.06    3  0.007191      0.1721
